# 01. Manifest와 grouped dataset 기초

목표: 1.66GB media를 받지 않고도 repository manifest, episode group, slice와 label coverage를 이해합니다. 아래 값은 revision `c9be8ed6...`의 Hugging Face API와 native metadata에서 검증했습니다.

In [ ]:
episodes = {
    '26047_Record004_260217': {'source_frames': 28, 'encoded_frames': 29, 'fps': 2.0365, 'duration': 14.24},
    '26047_Record050_260217': {'source_frames': 26, 'encoded_frames': 27, 'fps': 2.0517, 'duration': 13.16},
    '26047a_Record004_260217': {'source_frames': 30, 'encoded_frames': 31, 'fps': 1.5377, 'duration': 20.16},
    '26047a_Record084_260217': {'source_frames': 28, 'encoded_frames': 29, 'fps': 2.3387, 'duration': 12.40},
}
slices = [f'Camera_{index}' for index in range(6)] + ['point_cloud']
manifest = [(episode, media_slice) for episode in episodes for media_slice in slices]
print('groups:', len(episodes))
print('slices/group:', len(slices), slices)
print('samples:', len(manifest))
assert len(manifest) == 28

In [ ]:
video_samples = sum(1 for _, media_slice in manifest if media_slice.startswith('Camera_'))
scene_samples = sum(1 for _, media_slice in manifest if media_slice == 'point_cloud')
source_frames_per_camera = sum(item['source_frames'] for item in episodes.values())
encoded_frames_per_camera = sum(item['encoded_frames'] for item in episodes.values())
print({'video_samples': video_samples, '3d_samples': scene_samples})
print({'source_frames/camera': source_frames_per_camera, 'encoded_frames/camera': encoded_frames_per_camera})
assert (video_samples, scene_samples) == (24, 4)
assert source_frames_per_camera == 112 and encoded_frames_per_camera == 116

Source image 수와 encoded MP4 metadata의 frame 수가 episode마다 하나씩 다릅니다. Label join 전에 decode frame index와 `frames.json.frame_number`를 대조해야 합니다.

In [ ]:
repository_bytes = 1_659_090_565
pcd_sizes = [517_404_876, 522_183_516, 160_000_170, 160_000_154]
print(f'repository: {repository_bytes / 1_000_000_000:.2f} GB = {repository_bytes / 2**30:.2f} GiB')
print(f'PCD share: {sum(pcd_sizes) / repository_bytes:.1%}')
assert sum(pcd_sizes) / repository_bytes > 0.8

In [ ]:
frame_documents = 560  # Camera_0~4 × source frame 112
coverage = {'ground_truth': 270, 'hd_map': 560, 'trajectory': 112}
for field, count in coverage.items():
    print(f'{field:12s}: {count:3d}/{frame_documents} = {count / frame_documents:.1%}')
assert coverage['hd_map'] == frame_documents
assert coverage['trajectory'] == source_frames_per_camera

## 다음 질문

왜 `ground_truth` 비율을 270/896로 계산한 card의 약 30%와 이 notebook의 270/560=48.2%가 모두 가능한지 설명해 보세요. 분모가 source 전체 image인지, Voxel51 label frame document인지 먼저 정의해야 합니다.